# Eyetracker attention map overlap check

This notebook compares:
- `.npy` files present in `./Eyetracker_attention_maps/`
- `.npy` filenames referenced by `/home/csantiago/comparisons_df.pickle` (with fallbacks)

It reports overlap, missing files, and extras.


In [1]:
import os
import glob
from pathlib import Path
import pandas as pd

# --- Config ---
NpyDir = Path('/home/csantiago/Eyetracker_attention_maps/224x224')

pickle_candidates = [
    '/home/csantiago/comparisons_df.pickle',
    '/home/csantiago/comparisons.pickle',
    '/home/csantiago/comparisons.pkl',
    '/home/csantiago/comparisons_df.pkl',
]


print('NPY dir exists:', NpyDir.resolve(), NpyDir.exists())
for p in pickle_candidates:
    if Path(p).exists():
        print('Found pickle:', p)
        break
else:
    print('No pickle found in candidates list. Update `pickle_candidates`.')


NPY dir exists: /home/csantiago/Eyetracker_attention_maps/224x224 True
Found pickle: /home/csantiago/comparisons_df.pickle


In [2]:
# --- Load comparisons dataframe ---
pickle_path = None
for p in pickle_candidates:
    if Path(p).exists():
        pickle_path = p
        break

assert pickle_path is not None, 'Pickle file not found. Update `pickle_candidates`.'
df = pd.read_pickle(pickle_path)
print('Loaded:', pickle_path)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()


Loaded: /home/csantiago/comparisons_df.pickle
Shape: (13623, 9)
Columns: ['dataset', 'image_l', 'image_r', 'score', 'has_eyetracker', 'survey_id', 'trial_id', 'npy_file_l', 'npy_file_r']


,dataset,image_l,image_r,score,has_eyetracker,survey_id,trial_id,npy_file_l,npy_file_r
0,barcelona,381.jpg,1390.jpg,1,False,1684142210363,42,NaN,NaN
1,barcelona,418.jpg,1511.jpg,1,False,1684142210363,52,NaN,NaN
2,barcelona,544.jpg,3226.jpg,1,False,1684142210363,74,NaN,NaN
3,barcelona,374.jpg,5141.jpg,1,False,1684142210363,6,NaN,NaN
4,barcelona,435.jpg,5055.jpg,-1,False,1684142210363,69,NaN,NaN


In [3]:
# --- Identify columns that likely contain npy filenames/paths ---
cols_lower = {c: str(c).lower() for c in df.columns}
npy_cols = [c for c, lc in cols_lower.items() if ('npy' in lc) or ('eyetrack' in lc) or ('gaze' in lc)]

print('Candidate npy-related columns:', npy_cols)

# Heuristic: prefer specific left/right columns if they exist
preferred = []
for key in ['npy_file_l', 'npy_l', 'gaze_l', 'eyetrack_l', 'npy_left', 'left_npy']:
    for c in npy_cols:
        if str(c).lower() == key:
            preferred.append(c)
for key in ['npy_file_r', 'npy_r', 'gaze_r', 'eyetrack_r', 'npy_right', 'right_npy']:
    for c in npy_cols:
        if str(c).lower() == key:
            preferred.append(c)

if preferred:
    use_cols = preferred
else:
    # fallback to any npy-related cols
    use_cols = npy_cols

assert use_cols, (
    'No npy-related columns found. '\
    'Inspect df.columns and set `use_cols` manually in this cell.'
)
print('Using columns:', use_cols)


Candidate npy-related columns: ['has_eyetracker', 'npy_file_l', 'npy_file_r']
Using columns: ['npy_file_l', 'npy_file_r']


In [4]:
# --- Build expected set from dataframe ---
def to_basename(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    return os.path.basename(s)

expected = set()
for c in use_cols:
    series = df[c].map(to_basename)
    expected.update({v for v in series.dropna().unique().tolist() if v is not None})

# keep only npy-like names
expected = {v for v in expected if v.lower().endswith('.npy')}

print('Expected npy count (from pickle):', len(expected))
list(sorted(expected))[:10]


Expected npy count (from pickle): 2990


['surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial10_1881_left_eyetrack.npy',
 'surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial10_3289_right_eyetrack.npy',
 'surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial11_li_6589_2_left_eyetrack.npy',
 'surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial11_li_6589_5_right_eyetrack.npy',
 'surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial12_si_7073_0_left_eyetrack.npy',
 'surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial12_si_7073_6_right_eyetrack.npy',
 'surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial13_1883_left_eyetrack.npy',
 'surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial13_2473_right_eyetrack.npy',
 'surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536c

In [5]:
# --- Actual set from directory ---
assert NpyDir.exists(), f'NPY dir does not exist: {NpyDir.resolve()}'
actual_files = sorted(glob.glob(str(NpyDir / '*.npy')))
actual = {os.path.basename(p) for p in actual_files}

print('Actual npy count (in directory):', len(actual))
actual_files[:5]


Actual npy count (in directory): 2727


['/home/csantiago/Eyetracker_attention_maps/224x224/surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial10_1881_left_eyetrack.npy',
 '/home/csantiago/Eyetracker_attention_maps/224x224/surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial10_3289_right_eyetrack.npy',
 '/home/csantiago/Eyetracker_attention_maps/224x224/surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial11_li_6589_2_left_eyetrack.npy',
 '/home/csantiago/Eyetracker_attention_maps/224x224/surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial11_li_6589_5_right_eyetrack.npy',
 '/home/csantiago/Eyetracker_attention_maps/224x224/surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial12_si_7073_0_left_eyetrack.npy']

In [6]:
# --- Overlap analysis ---
overlap = expected & actual
missing = expected - actual
extra = actual - expected

def pct(a, b):
    return 0.0 if b == 0 else 100.0 * (a / b)

print('Overlap:', len(overlap))
print('Missing (expected but not found):', len(missing))
print('Extra (found but not expected):', len(extra))
print('Coverage of expected:', f"{pct(len(overlap), len(expected)):.2f}%")
print('Purity of actual:', f"{pct(len(overlap), len(actual)):.2f}%")


Overlap: 2727
Missing (expected but not found): 263
Extra (found but not expected): 0
Coverage of expected: 91.20%
Purity of actual: 100.00%


In [7]:
# --- Show samples ---
print('\nSample overlap (up to 20):')
for x in sorted(overlap)[:20]:
    print('  ', x)

print('\nSample missing (up to 20):')
for x in sorted(missing)[:20]:
    print('  ', x)

print('\nSample extra (up to 20):')
for x in sorted(extra)[:20]:
    print('  ', x)



Sample overlap (up to 20):
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial10_1881_left_eyetrack.npy
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial10_3289_right_eyetrack.npy
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial11_li_6589_2_left_eyetrack.npy
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial11_li_6589_5_right_eyetrack.npy
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial12_si_7073_0_left_eyetrack.npy
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial12_si_7073_6_right_eyetrack.npy
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial13_1883_left_eyetrack.npy
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564ffc3836a99e1333cd536cd9b1bdd_trial13_2473_right_eyetrack.npy
   surveycycling08ab6849b6ce9851d50c230e82c8b2ba0564

In [8]:
# --- Optional: save full lists to disk for inspection ---
out = Path('overlap_report')
out.mkdir(exist_ok=True)

(out / 'expected_from_pickle.txt').write_text('\n'.join(sorted(expected)) + '\n')
(out / 'actual_in_dir.txt').write_text('\n'.join(sorted(actual)) + '\n')
(out / 'overlap.txt').write_text('\n'.join(sorted(overlap)) + '\n')
(out / 'missing_expected.txt').write_text('\n'.join(sorted(missing)) + '\n')
(out / 'extra_actual.txt').write_text('\n'.join(sorted(extra)) + '\n')

print('Wrote reports to:', out.resolve())


Wrote reports to: /home/csantiago/survey_eye_tracker/overlap_report
